# Prodigy_Ablation_Study — R25 (18 esperimenti, analisi causale)

**Obiettivo**: diagnosticare il problema **T-tracking flat** osservato in R24F (best V08).

## Contesto e problema

Analisi R24F (93 run post-fix BUGS_2026-06-03) ha rivelato:
- ✅ Rete impara (val_total 0.169 highway, 0.189 mixed, 0.222 full)
- ✅ Prodigy V08 (cosine_no_restart) batte AdamW del 9-18%
- ⚠️ **Problema #1 CRITICO**: T predetto è una linea piatta intra-sample, NON segue T_true(t).
  La rete fa 'average estimation' cross-driver, non 'system identification' per-driver.
- ⚠️ Problema #2: v0/s0 saturano ai bound; a stuck a MIN.
- ⚠️ Problema #4: spike_rate 7-9%, sotto target FPGA [10%, 20%].

## Strategia

Ablation **one-at-a-time** rispetto al baseline V08 mixed (val_total best = 0.189), variando un singolo parametro per asse + un asse "COMBO" che mette insieme i 3 cambiamenti più promettenti dell'asse memoria.

**Tutti gli esperimenti**: stesso scenario **mixed**, stesso seed (42), stesso optimizer Prodigy V08, stesso budget computazionale (10 ep × 100 step) tranne asse E che lo varia. 

## 5 assi di ablation (18 run = 1 replica baseline + 17 ablation)

### Asse A — Memoria temporale (5 run — per problema #1 T-tracking)
| Run | seq_len | max_delay | bit_shift | Note |
|---|---:|---:|---:|---|
| A1_baseline_replica | 50 | 6 | 3 | = R24F V08 mixed (atteso val ≈ 0.189-0.20) |
| A2_seq_len_short | 30 | 6 | 3 | meno contesto temporale |
| A3_seq_len_long | 100 | 6 | 3 | più contesto, stessa memoria sinaptica |
| A4_delay_long | 50 | 18 | 3 | stessa seq, 3× memoria sinaptica |
| A5_leak_slow | 50 | 6 | 5 | LIF leak τ 4× più lungo |
| A6_COMBO_long_memory | 100 | 18 | 5 | TUTTI e 3 i cambiamenti insieme |

### Asse B — Loss balancing (3 run — per problema #1+#4 e gradient unbalance)
| Run | lambda_T_aux | Note |
|---|---:|---|
| B1_T_aux_low | 0.1 | Lieve supervisione diretta su T |
| B2_T_aux_mid | 1.0 | Pari peso a L_data |
| B3_T_aux_high | 10.0 | Forza la rete a guardare T |

### Asse C — Spike rate regularizer (3 run — per problema #5)
| Run | lambda_sr | Note |
|---|---:|---|
| C1_sr_off | 0.0 | Verifica che L_sr non contribuisca al pattern |
| C2_sr_high | 5.0 | 10× baseline, target ~15% spike rate |
| C3_sr_very_high | 20.0 | Estremo, capacità di spingere a 20% |

### Asse D — Capacity (3 run — per saturazione bound)
| Run | hidden_size | rank | Note |
|---|---:|---:|---|
| D1_small | 16 | 4 | sotto baseline (per stress test) |
| D2_mid | 64 | 16 | 2× baseline |
| D3_large | 128 | 32 | 4× baseline |

### Asse E — Training duration (3 run)
| Run | epochs | max_steps | Note |
|---|---:|---:|---|
| E1_short | 5 | 100 | metà tempo |
| E2_long | 20 | 100 | 2× tempo |
| E3_very_long | 30 | 100 | 3× tempo |

**Totale**: 1 baseline + 5 (A: A1 baseline è già contato; A2-A6 = 5 ablation) + 3 (B) + 3 (C) + 3 (D) + 3 (E) = **18 run**

Stima Azure: ~3-4h (vs 15h R24F).

> Nota: l'asse A include A1 come **baseline replica** (stesso config R24F V08 mixed), poi A2..A6 sono 5 vere ablation. Quindi la voce "baseline replica" e l'asse "A_memory" condividono A1.

## Tag naming

`R25_<axis>_<value>` es. `R25_A1_baseline_replica`, `R25_A3_seq_len_long`, `R25_B2_T_aux_mid`.

## Output diagnostici NEW (R25-specifici)

Telemetria estesa nei CSV (vedi `train.py` R25 changes):
- **15 gradient columns** in `training_batch_log.csv`: `gn_out_fc_<p>`, `gn_decoded_<p>`, `grad_dir_<p>` per ogni p ∈ {v0,T,s0,a,b}
- **11 tracking columns** in `training_log.csv`: `val_T_tracking_corr` (corr Pearson T_pred vs T_true), `val_<p>_pred_mean`, `val_<p>_intra_std` per ogni p

Plot diagnostici NEW (vedi `plot_diagnostics.py`):
- **G16**: gradient raw per canale (log scale, 5 curve)
- **G17**: gradient decoded post-sigmoid (log scale)
- **G18**: gradient direction sign mean per canale ([-1, +1])

## Reference

- `document/BUGS_2026-06-03.md` — 4 bug fix applicati
- `results/Prodigy_Study/MultiParam_PostFix/_aggregate_r24f.csv` — baseline da cui partiamo
- `Prodigy_MultiParam_Study_PostFix.ipynb` — notebook precedente (riferimento struttura)

In [ ]:
# Cell 1 -- Bootstrap + ENV check + verifica FIX + nuove R25 columns
import sys, os, subprocess, inspect
import importlib.util as _imu

# Install deps (idempotent)
for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Verifica file critici
for f in ['train.py', 'core/network.py', 'core/eventprop.py', 'config.py', 'data/generator.py',
          'document/BUGS_2026-06-03.md', 'utils/plot_diagnostics.py']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# ===== Verifica FIX BUGS_2026-06-03 (richiamo da R24F) =====
with open('core/network.py', encoding='utf-8') as fp:
    net_src = fp.read()
for marker in ('FIX-BUG-1', 'FIX-BUG-2', 'FIX-BUG-3', 'FIX-BUG-4'):
    assert marker in net_src, f'{marker} mancante in core/network.py'
assert 'raw / self.decode_scale' not in net_src, 'FIX#1 NON applicato'
print('[OK] 4 fix BUGS_2026-06-03 presenti')

# ===== Verifica R25 changes (CSV cols + CLI flag + new plot fn) =====
sys.path.insert(0, '.')
if 'train' in sys.modules: del sys.modules['train']  # forza reimport
from train import pinn_loss, CSVLogger, BatchCSVLogger
# Nuove colonne CSV
r25_csv_new = ['val_T_tracking_corr',
               'val_v0_pred_mean', 'val_T_pred_mean', 'val_s0_pred_mean', 'val_a_pred_mean', 'val_b_pred_mean',
               'val_v0_intra_std', 'val_T_intra_std', 'val_s0_intra_std', 'val_a_intra_std', 'val_b_intra_std']
for c in r25_csv_new:
    assert c in CSVLogger.COLS, f'CSV col mancante: {c}'
r25_batch_new = ['loss_T_aux',
                 'gn_out_fc_v0', 'gn_out_fc_T', 'gn_out_fc_s0', 'gn_out_fc_a', 'gn_out_fc_b',
                 'gn_decoded_v0', 'gn_decoded_T', 'gn_decoded_s0', 'gn_decoded_a', 'gn_decoded_b',
                 'grad_dir_v0', 'grad_dir_T', 'grad_dir_s0', 'grad_dir_a', 'grad_dir_b']
for c in r25_batch_new:
    assert c in BatchCSVLogger.COLS, f'Batch CSV col mancante: {c}'
print('[OK] R25: 11 col CSV + 16 col batch CSV presenti')

# pinn_loss signature
sig = inspect.signature(pinn_loss)
assert 'lam_T_aux' in sig.parameters
assert 'retain_params_grad' in sig.parameters
print('[OK] R25: pinn_loss signature con lam_T_aux + retain_params_grad')

# train.py CLI flags
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--prodigy_safeguard_warmup', '--prodigy_growth_rate', '--prodigy_d_coef',
             '--prodigy_betas', '--prodigy_use_bias_correction', '--prodigy_d0',
             '--prodigy_weight_decay', '--lambda_T_aux']:
    assert flag in help_txt, f'MISSING train.py flag: {flag}'
print('[OK] CLI flags Prodigy + R25 lambda_T_aux completi')

# Plot R25 functions importable
from utils.plot_diagnostics import (plot_g16_grad_raw_per_channel,
                                     plot_g17_grad_decoded_per_channel,
                                     plot_g18_grad_direction_per_channel)
print('[OK] G16/G17/G18 importable')

# git branch
br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch corrente: {br}')
assert br == 'Prodigy_Deep_Study', f'Wrong branch: {br}'
print('[OK] su branch Prodigy_Deep_Study, push per-run abilitato')

print('\nENV check passed (incluso R25 changes).')

In [ ]:
# Cell 2 -- Generate 18 EXPERIMENTS list (1 baseline replica + 17 ablation)
#
# Baseline V08 mixed da R24F (best mixed val=0.189):
#   Prodigy V08: lr=1.0, d_coef=1.0, d0=1e-6, growth=inf, scheduler=cosine_no_restart
#   scenario=mixed, hidden=32, rank=8, seq_len=50, max_delay=6, bit_shift=3 (default ALIF)
#   10 epoche × 100 step, lambda_sr=0.5
#
# NB: la versione V08 R24F era lr=0.5 (best mixed). Ma per coerenza con highway/full vincitore,
#     facciamo replica V08 lr=1.0 (best highway). val atteso ~0.20 mixed.

BASELINE = {
    'optimizer': 'prodigy', 'lr': 1.0,
    'd_coef': 1.0, 'd0': 1e-6, 'growth_rate': 'inf',
    'scheduler': 'cosine_no_restart',
    'scenario': 'mixed',
    'scenario_mix': 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    'cut_in_ratio': 0.0,
    'cache_path': 'data/cache_1500_mixed_cut0.0_ou0.0.pt',
    'hidden_size': 32, 'rank': 8,
    'seq_len': 50, 'max_delay': 6, 'bit_shift': 3,
    'epochs': 10, 'max_steps_per_epoch': 100,
    'lambda_sr': 0.5, 'lambda_T_aux': 0.0,
}

def _exp(tag, axis, desc, **overrides):
    """Crea esperimento sovrapponendo overrides al BASELINE."""
    e = dict(BASELINE)
    e.update(overrides)
    e['tag'] = tag
    e['axis'] = axis
    e['desc'] = desc
    return e

EXPERIMENTS = [
    # ===== Baseline replica (1 run) =====
    _exp('R25_A1_baseline_replica', 'BASELINE', 'Replica esatta V08 mixed (val atteso ~0.20)'),

    # ===== Asse A — Memoria temporale (5 run) =====
    _exp('R25_A2_seq_len_short',  'A_memory', 'seq_len 50→30 (meno contesto)', seq_len=30),
    _exp('R25_A3_seq_len_long',   'A_memory', 'seq_len 50→100 (più contesto)', seq_len=100),
    _exp('R25_A4_delay_long',     'A_memory', 'max_delay 6→18 (3× memoria sinaptica)', max_delay=18),
    _exp('R25_A5_leak_slow',      'A_memory', 'bit_shift 3→5 (leak τ 4× più lungo)', bit_shift=5),
    _exp('R25_A6_combo_memory',   'A_memory', 'COMBO: seq=100 + delay=18 + bit_shift=5',
         seq_len=100, max_delay=18, bit_shift=5),

    # ===== Asse B — Loss balancing (T aux supervision, 3 run) =====
    _exp('R25_B1_T_aux_low',      'B_loss', 'lambda_T_aux=0.1 (lieve)', lambda_T_aux=0.1),
    _exp('R25_B2_T_aux_mid',      'B_loss', 'lambda_T_aux=1.0 (pari L_data)', lambda_T_aux=1.0),
    _exp('R25_B3_T_aux_high',     'B_loss', 'lambda_T_aux=10.0 (forte)', lambda_T_aux=10.0),

    # ===== Asse C — Spike rate regularizer (3 run) =====
    _exp('R25_C1_sr_off',         'C_spike_rate', 'lambda_sr=0.0 (off)', lambda_sr=0.0),
    _exp('R25_C2_sr_high',        'C_spike_rate', 'lambda_sr=5.0 (10×)', lambda_sr=5.0),
    _exp('R25_C3_sr_very_high',   'C_spike_rate', 'lambda_sr=20.0 (estremo)', lambda_sr=20.0),

    # ===== Asse D — Capacity (3 run) =====
    _exp('R25_D1_small',          'D_capacity', 'hidden=16 rank=4 (sotto)', hidden_size=16, rank=4),
    _exp('R25_D2_mid',            'D_capacity', 'hidden=64 rank=16 (2×)', hidden_size=64, rank=16),
    _exp('R25_D3_large',          'D_capacity', 'hidden=128 rank=32 (4×)', hidden_size=128, rank=32),

    # ===== Asse E — Training duration (3 run) =====
    _exp('R25_E1_short',          'E_duration', 'epochs=5 (½ tempo)', epochs=5),
    _exp('R25_E2_long',           'E_duration', 'epochs=20 (2× tempo)', epochs=20),
    _exp('R25_E3_very_long',      'E_duration', 'epochs=30 (3× tempo)', epochs=30),
]

EXPECTED_N = 18  # 1 baseline + 5(A) + 3(B) + 3(C) + 3(D) + 3(E)
print(f'Total experiments: {len(EXPERIMENTS)} (atteso {EXPECTED_N})')
assert len(EXPERIMENTS) == EXPECTED_N, f'Expected {EXPECTED_N}, got {len(EXPERIMENTS)}'
from collections import Counter
axes_count = Counter(e['axis'] for e in EXPERIMENTS)
for a, n in sorted(axes_count.items()):
    print(f'  {a}: {n} run')
print(f'\nPrimo: {EXPERIMENTS[0]["tag"]}')
print(f'Ultimo: {EXPERIMENTS[-1]["tag"]}')

In [ ]:
# Cell 3 -- Cache check + AUTOGEN per mixed (R25 usa solo scenario mixed)
import os, time
import torch

cache_path = BASELINE['cache_path']
if os.path.isfile(cache_path):
    sz = os.path.getsize(cache_path) / 1024 / 1024
    print(f'[OK] mixed: {cache_path} ({sz:.1f} MB)')
else:
    print(f'[AUTOGEN] mixed: {cache_path} mancante, genero...')
    from data.generator import generate_dataset, parse_scenario_mix
    from config import SEED
    mix_dict = parse_scenario_mix(BASELINE['scenario_mix'])
    t0 = time.time()
    train_d = generate_dataset(1500, base_seed=SEED, scenario_mix=mix_dict,
                                cut_in_ratio=BASELINE['cut_in_ratio'], noise_scale=0.0)
    val_d   = generate_dataset(300,  base_seed=SEED+1, scenario_mix=mix_dict,
                                cut_in_ratio=BASELINE['cut_in_ratio'], noise_scale=0.0)
    torch.save({'train': train_d, 'val': val_d, 'seed': SEED}, cache_path)
    sz = os.path.getsize(cache_path) / 1024 / 1024
    print(f'  [SAVED] {cache_path} ({sz:.1f} MB) in {time.time()-t0:.0f}s')
print('\nCache ready.')

In [ ]:
# Cell 4 -- PRE-FLIGHT SANITY (R25 — verifica statica + empirica + smoke veloce)
import torch, inspect, sys, numpy as np
sys.path.insert(0, '.')
if 'core.network' in sys.modules: del sys.modules['core.network']
from core.network import build_model

print('=== PRE-FLIGHT 1: Static fix + R25 changes ===')
torch.manual_seed(42)
m = build_model('baseline')
src = inspect.getsource(m._decode_params)
assert 'raw / self.decode_scale' not in src
fc = m.layer_out.fc_weight
rm = fc.mean(dim=1).abs().max().item()
assert rm < 1e-6
assert sum(p.numel() for p in m.parameters()) == 864
print(f'  FIX#1/#2 OK; A1=864 params')

print('\n=== PRE-FLIGHT 2: Empirica (3 seeds) — saturation + gradient ===')
cache = torch.load(BASELINE['cache_path'], weights_only=False)
xs, ys = [], []
for s in cache['train'][:20]:
    x = s['x']; y = s['y']
    if isinstance(x, np.ndarray): x = torch.from_numpy(x).float()
    if isinstance(y, np.ndarray): y = torch.from_numpy(y).float()
    xs.append(x[:30]); ys.append(y[:30])
x = torch.stack(xs); y_seq = torch.stack(ys)

sat_max, grad_min, srs = [], [], []
for seed in [42, 123, 7]:
    torch.manual_seed(seed)
    m = build_model('baseline'); m.train()
    steps, sp = m.forward_sequence_with_stats(x)
    plo, phi = m.param_lo.view(1,1,-1), m.param_hi.view(1,1,-1)
    eps = 0.01*(phi-plo)
    sat = ((steps<plo+eps).float()+(steps>phi-eps).float()).mean(dim=(0,1)).cpu().numpy()
    sat_max.append(sat.max()); srs.append(sp.mean().item())
    loss = ((steps - (plo+phi)/2)**2).mean(); loss.backward()
    grad_min.append(m.layer_out.fc_weight.grad.abs().mean(dim=1).min().item())
print(f'  Sat max: {max(sat_max)*100:.2f}% | grad min: {min(grad_min):.5f} | spike: {np.mean(srs):.3f}')
assert max(sat_max) < 0.05 and min(grad_min) > 1e-5 and 0.03 <= np.mean(srs) <= 0.20

print('\n=== PRE-FLIGHT 3: Smoke train.py 1 ep × 5 step (verifica nuove col R25 popolate) ===')
import subprocess as sp, os, shutil
tag = '_R25_PREFLIGHT'
if os.path.isdir(f'checkpoints/{tag}'): shutil.rmtree(f'checkpoints/{tag}')
r = sp.run([sys.executable, 'train.py',
            '--training_method', 'baseline',
            '--epochs', '1', '--max_steps_per_epoch', '5',
            '--batch_size', '4', '--seq_len', '30',
            '--data_cache', BASELINE['cache_path'],
            '--n_train', '16', '--n_val', '8',
            '--tag', tag],
           capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print(r.stderr[-1500:])
assert r.returncode == 0, f'Smoke train.py FAILED rc={r.returncode}'
import pandas as pd
bdf = pd.read_csv(f'checkpoints/{tag}/training_batch_log.csv')
for c in ('gn_out_fc_T', 'gn_decoded_T', 'grad_dir_T', 'loss_T_aux'):
    assert c in bdf.columns, f'col {c} mancante nel batch log'
    assert bdf[c].notna().any(), f'col {c} tutta NaN'
print(f'  Smoke OK: {len(bdf)} batch logged, nuove col R25 popolate')
print(f'  gn_out_fc_T sample: {bdf["gn_out_fc_T"].iloc[-1]:.5f}')
print(f'  gn_decoded_T sample: {bdf["gn_decoded_T"].iloc[-1]:.5f}')
print(f'  grad_dir_T sample: {bdf["grad_dir_T"].iloc[-1]:+.3f}')
shutil.rmtree(f'checkpoints/{tag}', ignore_errors=True)

print('\n[OK] Pre-flight superato. Procedere con sweep 19 esperimenti.')

In [ ]:
# Cell 5 -- RUN sweep 18 esperimenti con PUSH per-run
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

SKIP_IF_EXISTS = True   # RESUME safe
RESULTS_DIR = 'results/Prodigy_Study/Ablation_R25'
os.makedirs(RESULTS_DIR, exist_ok=True)

_TMP_MSG = '/tmp/r25_msg.txt' if os.path.isdir('/tmp') else 'r25_msg.txt'

def _build_cli(e):
    cli = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', str(e['epochs']),
        '--max_steps_per_epoch', str(e['max_steps_per_epoch']),
        '--batch_size', '8', '--val_batch_size', '32',
        '--seq_len', str(e['seq_len']),
        '--cf_hidden_size', str(e['hidden_size']),
        '--cf_rank', str(e['rank']),
        '--cf_max_delay', str(e['max_delay']),
        '--cf_bit_shift', str(e['bit_shift']),
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', str(e['lambda_sr']),
        '--lambda_T_aux', str(e['lambda_T_aux']),
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--data_cache', e['cache_path'],
        '--optimizer', e['optimizer'],
        '--lr', str(e['lr']), '--max_lr', str(e['lr']),
        '--scheduler', e['scheduler'],
        '--prodigy_betas', '0.9,0.99',
        '--prodigy_d_coef', str(e['d_coef']),
        '--prodigy_d0', str(e['d0']),
        '--prodigy_weight_decay', '0.01',
        '--prodigy_use_bias_correction', '1',
        '--prodigy_safeguard_warmup', '1',
        '--prodigy_growth_rate', str(e['growth_rate']),
        '--tag', e['tag']]
    return cli

def _push_run(e):
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst_parent = f'{RESULTS_DIR}/{e["axis"]}'
    dst = f'{dst_parent}/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante, skip')
        return False
    os.makedirs(dst_parent, exist_ok=True)
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    log_path = f'{dst}/training_log.csv'
    if os.path.isfile(log_path):
        try:
            edf = pd.read_csv(log_path)
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                tcorr = edf.get('val_T_tracking_corr', pd.Series([float('nan')]))
                tcorr_at_best = tcorr.iloc[bi] if len(tcorr) > bi else float('nan')
                val_str = (f'best val_total={edf.val_total.min():.4f} val_data={edf.val_data.iloc[bi]:.4f} '
                           f'T_corr={tcorr_at_best:.3f} (E{bi+1}/{len(edf)})')
        except Exception as ex:
            val_str = f'(log read failed: {ex})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (R25 Ablation): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'axis={e["axis"]} desc={e["desc"]}\n'
        f'Branch Prodigy_Deep_Study | post BUGS_2026-06-03 fix\n'
    )
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                print(f'   [push] nothing to commit')
                return True
            print(f'   [push] commit failed rc={r.returncode}: {r.stderr[-300:]}')
            return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin','Prodigy_Deep_Study'],
                       capture_output=True, text=True)
        r2 = subprocess.run(['git','push','origin','Prodigy_Deep_Study'], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push] push failed rc={r2.returncode}: {r2.stderr[-300:]}')
            return False
        print(f'   [push] OK')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

# Esecuzione
run_results = []
t_start = time.time()
total = len(EXPERIMENTS)
for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst_log = f'{RESULTS_DIR}/{e["axis"]}/{tag}/training_log.csv'
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val_total={edf.val_total.min():.4f}'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[{i}/{total}] [SKIP_EXIST] {tag}: {v_str}')
        run_results.append({'tag': tag, 'status':'skipped'})
        continue
    print(f'\n{"="*78}\n[{i}/{total}] {tag}: {e["desc"]}\n   axis={e["axis"]}\n{"="*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(e), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    print(f'   pushing {tag}...')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'status': status, 'pushed': pushed, 'elapsed': el})

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results[-5:]:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<45} {r['status']:<10} push={r.get('pushed','N/A')}")

In [ ]:
# Cell 6 -- Aggregate analysis con focus tracking_corr + gradient diagnostics
import pandas as pd, os, math
RESULTS_DIR = 'results/Prodigy_Study/Ablation_R25'

rows = []
for e in EXPERIMENTS:
    tag = e['tag']
    base = f'{RESULTS_DIR}/{e["axis"]}/{tag}'
    log = f'{base}/training_log.csv'
    blog = f'{base}/training_batch_log.csv'
    row_base = {'tag': tag, 'axis': e['axis'], 'desc': e['desc']}
    if not os.path.isfile(log):
        rows.append({**row_base, 'status': 'MISSING'})
        continue
    df = pd.read_csv(log)
    bdf = pd.read_csv(blog) if os.path.isfile(blog) else None
    if len(df) == 0:
        rows.append({**row_base, 'status': 'EMPTY'})
        continue
    bi = int(df['val_total'].idxmin())
    row = {**row_base, 'status': 'OK', 'n_ep': len(df),
           'val_total_best': float(df['val_total'].iloc[bi]),
           'val_data_best':  float(df['val_data'].iloc[bi]),
           'val_T_tracking_corr_best': float(df.get('val_T_tracking_corr', pd.Series([float('nan')])).iloc[bi]) if 'val_T_tracking_corr' in df.columns else float('nan'),
           'val_T_intra_std_best':    float(df.get('val_T_intra_std', pd.Series([float('nan')])).iloc[bi]) if 'val_T_intra_std' in df.columns else float('nan'),
           'spike_rate_avg': float(df['spike_rate'].mean()),
           'best_ep': bi+1,
    }
    # Per-channel pred mean al best epoch (saturation indicator)
    for pn in ('v0','T','s0','a','b'):
        col_pm = f'val_{pn}_pred_mean'
        if col_pm in df.columns:
            row[col_pm] = float(df[col_pm].iloc[bi])
    # Gradient stats medi su last 50% batch (skip noise transient)
    if bdf is not None and len(bdf) > 10:
        half = len(bdf) // 2
        tail = bdf.iloc[half:]
        for pn in ('v0','T','s0','a','b'):
            for prefix in ('gn_out_fc', 'gn_decoded', 'grad_dir'):
                col = f'{prefix}_{pn}'
                if col in tail.columns:
                    vals = tail[col].dropna()
                    if len(vals) > 0:
                        row[f'{col}_mean'] = float(vals.mean())
    rows.append(row)

df_all = pd.DataFrame(rows)
n_ok = (df_all.status=='OK').sum()
print(f'Runs: {len(df_all)} | OK: {n_ok}')
os.makedirs(RESULTS_DIR, exist_ok=True)
df_all.to_csv(f'{RESULTS_DIR}/_aggregate_r25.csv', index=False)
print(f'[saved] {RESULTS_DIR}/_aggregate_r25.csv')

# Baseline = R25_A1
baseline_row = df_all[df_all.tag == 'R25_A1_baseline_replica']
baseline_val = float(baseline_row.val_total_best.iloc[0]) if len(baseline_row) > 0 and baseline_row.status.iloc[0] == 'OK' else float('nan')
baseline_Tcorr = float(baseline_row.val_T_tracking_corr_best.iloc[0]) if len(baseline_row) > 0 and baseline_row.status.iloc[0] == 'OK' else float('nan')
print(f'\nBASELINE val_total={baseline_val:.4f}  T_tracking_corr={baseline_Tcorr:.3f}')

# Delta tabella
df_all['delta_val_total'] = df_all['val_total_best'] - baseline_val
df_all['delta_T_corr']    = df_all['val_T_tracking_corr_best'] - baseline_Tcorr

from IPython.display import display
print('\n== TUTTI I RUN — delta vs baseline ==')
show_cols = ['tag','axis','val_total_best','delta_val_total','val_T_tracking_corr_best','delta_T_corr',
             'spike_rate_avg','best_ep']
display(df_all[df_all.status=='OK'][show_cols].round(4))

In [ ]:
# Cell 7 -- Pareto plots per asse: delta_val_total vs delta_T_corr
import matplotlib.pyplot as plt, os
RESULTS_DIR = 'results/Prodigy_Study/Ablation_R25'
os.makedirs(f'{RESULTS_DIR}/_summary_plots', exist_ok=True)

ok = df_all[df_all.status=='OK'].copy()
axes = ['BASELINE', 'A_memory', 'B_loss', 'C_spike_rate', 'D_capacity', 'E_duration']
colors = {'BASELINE':'black', 'A_memory':'tab:blue', 'B_loss':'tab:orange',
          'C_spike_rate':'tab:green', 'D_capacity':'tab:purple', 'E_duration':'tab:red'}

fig, ax = plt.subplots(figsize=(11, 7))
for axis in axes:
    sub = ok[ok.axis == axis]
    if len(sub) == 0: continue
    ax.scatter(sub.delta_T_corr, sub.delta_val_total,
               s=100 if axis == 'BASELINE' else 80, alpha=0.85,
               color=colors[axis], label=axis, edgecolors='black' if axis=='BASELINE' else None)
    for _, r in sub.iterrows():
        ax.annotate(r.tag.replace('R25_',''), (r.delta_T_corr, r.delta_val_total),
                    fontsize=7, alpha=0.8, xytext=(4, 4), textcoords='offset points')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Δ T_tracking_corr (vs baseline) — destra = T traccia meglio')
ax.set_ylabel('Δ val_total_best (vs baseline) — basso = val migliore')
ax.set_title('R25 Ablation Pareto: improvement T-tracking vs val_total\n(quadrante in basso a destra = WIN)')
ax.legend(title='Asse', loc='best', fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.95, 0.02, 'WIN ↘', transform=ax.transAxes, ha='right', va='bottom',
        fontsize=20, color='green', alpha=0.4, fontweight='bold')
ax.text(0.05, 0.97, 'LOSS ↖', transform=ax.transAxes, ha='left', va='top',
        fontsize=20, color='red', alpha=0.4, fontweight='bold')
plt.tight_layout()
out = f'{RESULTS_DIR}/_summary_plots/pareto_T_corr_vs_val.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out}')

# Per asse: bar plot delta T_corr
fig, axarr = plt.subplots(1, 2, figsize=(16, 5))
for j, (metric, ylabel, title) in enumerate([
    ('delta_T_corr', 'Δ T_tracking_corr', 'Miglioramento T-tracking per run (>0 = meglio)'),
    ('delta_val_total', 'Δ val_total', 'Miglioramento val_total per run (<0 = meglio)')
]):
    ax = axarr[j]
    for axis in axes:
        sub = ok[ok.axis == axis].sort_values('tag')
        if len(sub) == 0: continue
        x = sub.tag.str.replace('R25_', '')
        ax.bar(x, sub[metric], color=colors[axis], alpha=0.8, label=axis if j == 0 else None)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.tick_params(axis='x', rotation=70, labelsize=7)
    ax.grid(alpha=0.3, axis='y')
axarr[0].legend(fontsize=8)
plt.tight_layout()
out2 = f'{RESULTS_DIR}/_summary_plots/per_run_bars.png'
fig.savefig(out2, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out2}')

In [ ]:
# Cell 8 -- Diagnostic G16/G17/G18 + G7 violin per TOP 3 run vs BASELINE
# Carica i plot pre-renderizzati (già nei push per-run) e mostra side-by-side.
from IPython.display import Image, display, Markdown
import os

ok = df_all[df_all.status=='OK'].copy()
if 'val_T_tracking_corr_best' in ok.columns:
    # TOP 3 per delta_T_corr POSITIVO (= maggior miglioramento tracking T)
    top_T = ok.sort_values('delta_T_corr', ascending=False).head(3)
    display(Markdown('## TOP 3 per miglioramento T-tracking'))
    display(top_T[['tag','axis','desc','delta_T_corr','delta_val_total']].round(4))

    for _, r in top_T.iterrows():
        tag = r['tag']; axis = r['axis']
        display(Markdown(f'### {tag} ({axis}) — {r["desc"]}'))
        for plot in ('G7_violin_params.png', 'G16_grad_raw_per_ch.png',
                     'G17_grad_decoded_per_ch.png', 'G18_grad_direction_per_ch.png',
                     'G13_traj_highway.png'):
            p = f'{RESULTS_DIR}/{axis}/{tag}/plots/{plot}'
            if os.path.isfile(p):
                display(Image(p, width=950))
            else:
                print(f'   [MISSING] {p}')

# Baseline diagnostics confronto
display(Markdown('## BASELINE (R25_A1_baseline_replica) — riferimento'))
for plot in ('G7_violin_params.png', 'G16_grad_raw_per_ch.png',
             'G17_grad_decoded_per_ch.png', 'G18_grad_direction_per_ch.png',
             'G13_traj_highway.png'):
    p = f'{RESULTS_DIR}/BASELINE/R25_A1_baseline_replica/plots/{plot}'
    if os.path.isfile(p):
        display(Image(p, width=950))

In [ ]:
# Cell 9 -- Riepilogo + raccomandazioni
from IPython.display import display, Markdown

if (df_all.status == 'OK').sum() == 0:
    print('Nessun run OK — saltare riepilogo.')
else:
    ok = df_all[df_all.status=='OK'].copy()
    best_T  = ok.sort_values('delta_T_corr', ascending=False).iloc[0]
    best_v  = ok.sort_values('delta_val_total').iloc[0]
    display(Markdown(f"""
## Risultati R25 — sintesi

**Baseline**: `R25_A1_baseline_replica` → val={baseline_val:.4f}, T_corr={baseline_Tcorr:.3f}

### Best per T-tracking improvement
- **{best_T.tag}** ({best_T.axis}): T_corr {baseline_Tcorr:.3f} → {best_T.val_T_tracking_corr_best:.3f}  (Δ = **{best_T.delta_T_corr:+.3f}**)
  - val_total {baseline_val:.4f} → {best_T.val_total_best:.4f}  (Δ = {best_T.delta_val_total:+.4f})
  - {best_T.desc}

### Best per val_total
- **{best_v.tag}** ({best_v.axis}): val {baseline_val:.4f} → {best_v.val_total_best:.4f}  (Δ = **{best_v.delta_val_total:+.4f}**)
  - T_corr {baseline_Tcorr:.3f} → {best_v.val_T_tracking_corr_best:.3f}  (Δ = {best_v.delta_T_corr:+.3f})
  - {best_v.desc}

### Per asse: best e mean
"""))
    agg = ok.groupby('axis').agg(
        n=('tag','count'),
        val_min=('val_total_best','min'),
        val_mean=('val_total_best','mean'),
        T_corr_max=('val_T_tracking_corr_best','max'),
        T_corr_mean=('val_T_tracking_corr_best','mean'),
    ).round(4)
    display(agg)
    display(Markdown('### Note\n\n'
                     '- Δ T_corr positivo significa T predetto traccia meglio T_true dinamico\n'
                     '- Δ val_total negativo significa loss minore (miglioramento)\n'
                     '- WIN ideale: Δ T_corr ↑ + Δ val_total ↓'))